# Task 3 — Consultas analíticas e dashboard

Consome o star schema da Task 2 via Amazon Athena + dashboard interativo Jupyter.

Portátil local↔SageMaker: profile via env var (`AWS_PROFILE=projetos` local; sem env no SageMaker → usa role).

## 0 — Setup

In [1]:
# %pip install -q awswrangler pandas seaborn matplotlib ipywidgets pyarrow

In [2]:
import os
from pathlib import Path

import awswrangler as wr
import boto3
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from ipywidgets import (
    Dropdown, IntSlider, SelectMultiple, SelectionRangeSlider,
    VBox, HBox, Output, interactive_output,
)
from IPython.display import display

sns.set_theme(style="whitegrid")

In [3]:
# Lê src/.env (gerado por Terraform) se rodar local. No SageMaker pula.
env_path = Path("../src/.env")
if env_path.exists():
    for line in env_path.read_text().splitlines():
        if "=" in line and not line.startswith("#"):
            k, _, v = line.partition("=")
            os.environ.setdefault(k.strip(), v.strip())

GLUE_DATABASE          = os.environ.get("GLUE_DATABASE", "classicmodels_analytics")
ATHENA_WORKGROUP       = os.environ.get("ATHENA_WORKGROUP", "classicmodels-lab")
ATHENA_OUTPUT_LOCATION = os.environ.get(
    "ATHENA_OUTPUT_LOCATION",
    "s3://lab-classicmodels-gt-projetos-athena-results/results/",
)

# Profile: força "projetos" local (default config quebrada com region inválida).
# No SageMaker, exportar IN_SAGEMAKER=1 para cair na role automática.
if os.environ.get("IN_SAGEMAKER") == "1":
    AWS_PROFILE = None
else:
    AWS_PROFILE = os.environ.get("AWS_PROFILE", "projetos")

AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")

boto3_session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)

print(f"GLUE_DATABASE          = {GLUE_DATABASE}")
print(f"ATHENA_WORKGROUP       = {ATHENA_WORKGROUP}")
print(f"ATHENA_OUTPUT_LOCATION = {ATHENA_OUTPUT_LOCATION}")
print(f"AWS_PROFILE            = {AWS_PROFILE or '(role)'}")
print(f"AWS_REGION             = {AWS_REGION}")

GLUE_DATABASE          = classicmodels_analytics
ATHENA_WORKGROUP       = classicmodels-lab
ATHENA_OUTPUT_LOCATION = s3://lab-classicmodels-gt-projetos-athena-results/results/
AWS_PROFILE            = projetos
AWS_REGION             = us-east-1


In [4]:
def run_query(sql: str) -> pd.DataFrame:
    """Executa SQL no Athena e retorna DataFrame."""
    return wr.athena.read_sql_query(
        sql=sql,
        database=GLUE_DATABASE,
        workgroup=ATHENA_WORKGROUP,
        s3_output=ATHENA_OUTPUT_LOCATION,
        boto3_session=boto3_session,
    )

## 4.2 — Consulta exploratória em `dim_products`

In [5]:
sql_products = """
SELECT
    product_id,
    product_name,
    product_line,
    product_vendor
FROM dim_products
LIMIT 20
"""

df_products = run_query(sql_products)
df_products.head()

,product_id,product_name,product_line,product_vendor
0,S12_1666,1958 Setra Bus,Trucks and Buses,Welly Diecast Productions
1,S12_4473,1957 Chevy Pickup,Trucks and Buses,Exoto Designs
2,S18_1097,1940 Ford Pickup Truck,Trucks and Buses,Studio M Art Models
3,S18_2319,1964 Mercedes Tour Bus,Trucks and Buses,Unimax Art Galleries
4,S18_2432,1926 Ford Fire Engine,Trucks and Buses,Carousel DieCast Legends


## 4.3 — Vendas totais por país

In [6]:
sql_country = """
SELECT
    dim_countries.country,
    SUM(fact_orders.sales_amount) AS total_sales
FROM fact_orders
JOIN dim_countries ON fact_orders.country_key = dim_countries.country_key
GROUP BY dim_countries.country
ORDER BY total_sales DESC
LIMIT 10
"""

df_country = run_query(sql_country)
df_country

,country,total_sales
0,Spain,17590225.44
1,USA,13093120.20
2,Singapore,9503920.08
3,France,4029496.08
4,Germany,3143535.84
5,Australia,2250330.36
6,New Zealand,1907388.04
7,UK,1747789.76
8,Switzerland,1740446.72
9,Italy,1442467.24


## 4.4 — Detalhamento (data, linha, produto, país)

In [7]:
sql_detail = """
SELECT
    dim_dates.full_date,
    dim_products.product_line,
    dim_products.product_name,
    dim_countries.country,
    SUM(fact_orders.sales_amount) AS total_sales
FROM fact_orders
JOIN dim_products  ON fact_orders.product_id     = dim_products.product_id
JOIN dim_countries ON fact_orders.country_key    = dim_countries.country_key
JOIN dim_dates     ON fact_orders.order_date_key = dim_dates.date_key
GROUP BY
    dim_dates.full_date,
    dim_products.product_line,
    dim_products.product_name,
    dim_countries.country
"""

df_detail = run_query(sql_detail)
df_detail["full_date"]   = pd.to_datetime(df_detail["full_date"])
df_detail["total_sales"] = df_detail["total_sales"].astype(float)
print(f"linhas: {len(df_detail)}")
print(df_detail.dtypes)
df_detail.head()

linhas: 2996
full_date       datetime64[s]
product_line           string
product_name           string
country                string
total_sales           float64
dtype: object


,full_date,product_line,product_name,country,total_sales
0,2005-05-29,Vintage Cars,1917 Grand Touring Sedan,Australia,90576.0
1,2005-05-29,Classic Cars,1970 Chevy Chevelle SS 454,Australia,57849.6
2,2005-05-29,Vintage Cars,1939 Chevrolet Deluxe Coupe,Australia,23176.8
3,2003-07-16,Vintage Cars,1934 Ford V8 Coupe,Australia,16790.4
4,2003-07-16,Vintage Cars,1917 Maxwell Touring Car,Australia,39048.0


## 4.5 — Dashboard interativo

Filtros: intervalo de datas, país, linha de produto, Top N. Gráfico de barras horizontais com Top N produtos.

In [ ]:
dates_sorted = sorted(df_detail["full_date"].dt.date.unique())
countries    = sorted(df_detail["country"].unique())
product_lines = sorted(df_detail["product_line"].unique())

date_range = SelectionRangeSlider(
    options=[(d.isoformat(), d) for d in dates_sorted],
    index=(0, len(dates_sorted) - 1),
    description="Datas:",
    continuous_update=False,
    layout={"width": "600px"},
)
country_sel = SelectMultiple(
    options=countries, value=tuple(countries),
    description="Países:", rows=6,
)
line_sel = SelectMultiple(
    options=product_lines, value=tuple(product_lines),
    description="Linha:", rows=6,
)
top_n = IntSlider(value=10, min=1, max=10, description="Top N:")

def update(date_range, country_sel, line_sel, top_n):
    d0, d1 = pd.to_datetime(date_range[0]), pd.to_datetime(date_range[1])
    df = df_detail[
        (df_detail["full_date"] >= d0) &
        (df_detail["full_date"] <= d1) &
        (df_detail["country"].isin(country_sel)) &
        (df_detail["product_line"].isin(line_sel))
    ]
    if df.empty:
        print("Sem dados para filtros selecionados.")
        return
    top = (
        df.groupby("product_name", as_index=False)["total_sales"]
          .sum()
          .nlargest(top_n, "total_sales")
    )
    fig, ax = plt.subplots(figsize=(10, max(3, 0.5 * len(top))))
    sns.barplot(data=top, x="total_sales", y="product_name", ax=ax, palette="viridis")
    ax.set_xlabel("Vendas totais (sales_amount)")
    ax.set_ylabel("Produto")
    ax.set_title(f"Top {top_n} produtos — {d0.date()} a {d1.date()}")
    plt.tight_layout()
    plt.show()

out = interactive_output(
    update,
    {
        "date_range":  date_range,
        "country_sel": country_sel,
        "line_sel":    line_sel,
        "top_n":       top_n,
    },
)

display(VBox([date_range, HBox([country_sel, line_sel]), top_n, out]))